In [1]:
# @title Jurafsky_Style_Minimum_Edit_Distance_Table_Generator

# ============================================================
# IMPORT LIBRARIES
# ============================================================

# ipywidgets → used to build interactive UI elements
import ipywidgets as widgets

# HTML → allows styled table rendering
from IPython.display import display, HTML



# ============================================================
# FUNCTION 1: BUILD DP TABLE + STORE CANDIDATES
# ============================================================
def compute_med(source, target, ins_cost=1, del_cost=1, sub_cost=2):
    """
    Builds the Dynamic Programming table for
    Jurafsky-style Minimum Edit Distance.

    Also stores the candidate values:
        (left+ins, up+del, diag+cost)
    for display purposes.
    """

    m = len(source)
    n = len(target)

    # Create empty DP table
    D = [[0]*(n+1) for _ in range(m+1)]

    # Dictionary to store candidate triples
    candidates = {}

    # -------------------------------
    # STEP 1: Initialize bottom row
    # -------------------------------
    for j in range(1, n+1):
        D[0][j] = D[0][j-1] + ins_cost

    # -------------------------------
    # STEP 2: Initialize first column
    # -------------------------------
    for i in range(1, m+1):
        D[i][0] = D[i-1][0] + del_cost

    # -------------------------------
    # STEP 3: Fill rest of table
    # -------------------------------
    for i in range(1, m+1):
        for j in range(1, n+1):

            # Determine substitution cost
            match_cost = 0 if source[i-1] == target[j-1] else sub_cost

            # Compute three candidate moves
            left  = D[i][j-1] + ins_cost
            up    = D[i-1][j] + del_cost
            diag  = D[i-1][j-1] + match_cost

            # Store minimum in table
            D[i][j] = min(left, up, diag)

            # Save candidate triple
            candidates[(i,j)] = (left, up, diag)

    return D, candidates



# ============================================================
# FUNCTION 2: TRACE OPTIMAL PATH
# ============================================================
def trace_path(source, target, D, ins_cost=1, del_cost=1, sub_cost=2):
    """
    Backtraces from top-right cell to (0,0)
    to determine optimal path and arrow positions.
    """

    m, n = len(source), len(target)

    path_cells = set()
    arrow_cells = {}

    i, j = m, n

    while i > 0 or j > 0:

        path_cells.add((i, j))

        if i == 0:
            arrow_cells[(i, j-1)] = '→'
            j -= 1

        elif j == 0:
            arrow_cells[(i-1, j)] = '↑'
            i -= 1

        else:
            match_cost = 0 if source[i-1] == target[j-1] else sub_cost

            left  = D[i][j-1] + ins_cost
            up    = D[i-1][j] + del_cost
            diag  = D[i-1][j-1] + match_cost

            best = min(left, up, diag)

            if best == diag:
                arrow_cells[(i-1, j-1)] = '↗'
                i -= 1
                j -= 1
            elif best == up:
                arrow_cells[(i-1, j)] = '↑'
                i -= 1
            else:
                arrow_cells[(i, j-1)] = '→'
                j -= 1

    path_cells.add((0,0))

    return path_cells, arrow_cells



# ============================================================
# FUNCTION 3: DISPLAY TABLE
# ============================================================
def display_jurafsky_table(source, target, D, candidates,
                           ins_cost=1, del_cost=1, sub_cost=2):

    m, n = len(source), len(target)

    path_cells, arrow_cells = trace_path(
        source, target, D, ins_cost, del_cost, sub_cost
    )

    html = """
    <style>
        .med-table {
            border-collapse: collapse;
            font-family: monospace;
            font-size: 14px;
            margin-top: 15px;
        }
        .med-table td {
            border: 1px solid #444;
            padding: 6px 10px;
            text-align: center;
            min-width: 60px;
        }
        .candidate {
            font-size: 10px;
            color: #444;
        }
    </style>
    <table class="med-table">
    """

    # Print rows from TOP → BOTTOM
    for i in range(m, -1, -1):

        label = "#" if i == 0 else source[i-1]

        html += "<tr>"
        html += f"<td><b>{label}</b></td>"

        for j in range(n+1):

            style = ""

            # Light blue for optimal path
            if (i, j) in path_cells:
                style = "background-color:#d9f0ff; font-weight:bold;"

            cell_content = ""

            # Show candidate triple if available
            if (i, j) in candidates:
                left, up, diag = candidates[(i,j)]
                cell_content += f"<div class='candidate'>({left},{up},{diag})</div>"

            value = str(D[i][j])

            # Add arrow on SOURCE cell
            if (i, j) in arrow_cells:
                value += " " + arrow_cells[(i, j)]

            cell_content += f"<div>{value}</div>"

            html += f'<td style="{style}">{cell_content}</td>'

        html += "</tr>"

    # Horizontal string row
    html += "<tr><td></td><td><b>#</b></td>"
    for ch in target:
        html += f"<td><b>{ch}</b></td>"
    html += "</tr>"

    html += "</table>"

    display(HTML(html))
    print(f"Final Edit Distance = {D[m][n]} (TOP-RIGHT cell)")
    print("-"*70)



# ============================================================
# USER INTERFACE SECTION
# ============================================================

# Input boxes
source_input = widgets.Text(description='Source:')
target_input = widgets.Text(description='Target:')
ins_input    = widgets.IntText(value=1, description='Insertion Cost:')
del_input    = widgets.IntText(value=1, description='Deletion Cost:')
sub_input    = widgets.IntText(value=2, description='Substitution Cost:')

# Buttons
generate_button = widgets.Button(description="Generate Table")
clear_button    = widgets.Button(description="Clear All Tables")

# Persistent output area
out = widgets.Output()



# ------------------------------------------------------------
# WHAT HAPPENS WHEN "Generate Table" IS CLICKED
# ------------------------------------------------------------
def on_generate_click(b):

    source = source_input.value
    target = target_input.value
    ins    = ins_input.value
    dele   = del_input.value
    sub    = sub_input.value

    D, candidates = compute_med(source, target, ins, dele, sub)

    # IMPORTANT:
    # We DO NOT clear output.
    # So previous tables remain visible.
    with out:
        display_jurafsky_table(source, target, D, candidates, ins, dele, sub)

    # Clear input fields for next entry
    source_input.value = ""
    target_input.value = ""


# ------------------------------------------------------------
# WHAT HAPPENS WHEN "Clear All Tables" IS CLICKED
# ------------------------------------------------------------
def on_clear_click(b):
    out.clear_output()


# Connect buttons to their functions
generate_button.on_click(on_generate_click)
clear_button.on_click(on_clear_click)


# Display UI
display(source_input, target_input,
        ins_input, del_input, sub_input,
        generate_button, clear_button,
        out)

Text(value='', description='Source:')

Text(value='', description='Target:')

IntText(value=1, description='Insertion Cost:')

IntText(value=1, description='Deletion Cost:')

IntText(value=2, description='Substitution Cost:')

Button(description='Generate Table', style=ButtonStyle())

Button(description='Clear All Tables', style=ButtonStyle())

Output()